### **Extraer el `post`, `reacciones`, `n° comentarios`, `n° compartidas` y `url` de un perfil público de `Facebook`**

Vamos a scrapear el siguiente sitio:

https://www.facebook.com/elcorteingles

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/mgF22Gwh/ws-351.png"></center>

Debo localizar el boton de cerrar el recuadro:

<center><img src="https://i.postimg.cc/xCs3XVPN/ws-352.png"></center>

Entramos al perfil:

<center><img src="https://i.postimg.cc/QMQ81kv3/ws-350.png"></center>

Localizamos el recuadro que almacene toda la información de cada uno de los posts (texto, likes, reacciones, n° comentarios, n° compartidas, url)

<center><img src="https://i.postimg.cc/qMFvT729/ws-349.png"></center>


#### **Código**

Por tanto, esccribimos el siguiente código:

In [56]:
from time import sleep
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import NoSuchElementException
#from webdriver_manager.chrome import ChromeDriverManager  # pip install webdriver-manager


# Funcion para hacer un Scrolling SUAVIZADO dependiendo de cuantos scrollings ya he hecho
# Mientras mas escrolls llevo dando, mas pixeles voy bajando
# Para esto utilizo el scrolling que voy haciendo actualmente para bajar hasta cierta posicion en la pagina
def hacer_scrolling_suavizado(driver, iteracion):
    bajar_hasta = 2000 * (iteracion + 1)
    inicio = (iteracion * 2000) # Inicio donde termine la anterior iteracion
    for i in range(inicio,  bajar_hasta, 5): # Cada vez avanzo 5 pixeles
        scrollingScript = f""" 
          window.scrollTo(0, {i})
        """
        driver.execute_script(scrollingScript)


opts = Options()
opts.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36")
# opts.add_argument("--headless") # Headless Mode

#URL SEMILLA
website = "https://www.facebook.com/elcorteingles"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
# driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options=options)
driver = webdriver.Chrome(options=opts)
driver.get(website)
driver.maximize_window()

# Dejamos que el sitio cargue por 3 segundos
sleep(3)

# Cerramos el diálogo que nos pida hacer Login
# Si necesitaramos hacer Login, tendriamos que llenar el formulario
cerrar_dialogo = driver.find_element(by=By.XPATH, value='//div[@role="dialog"]//div[@aria-label="Cerrar"]')
cerrar_dialogo.click()
sleep(0.5)

# Pulsar el boton de cookies
try:
    btn = driver.find_element(by=By.XPATH, value='//div[not(@aria-disabled) and @aria-label="Allow all cookies"]')
    btn.click()
    sleep(0.5)
except:
    pass

# La estrategia para la extracción de los posts será hacer scrolling tantas veces como sea necesaria
# para obtener al menos 50 posts. Con un máximo de 100 scrolls para evitar que se cuelgue infinitamente
# en caso de que no hayan suficientes posts
# Hay otras estrategias que también podriamos utilizar:
#   1. Sabiendo que en cada scroll se cargan 3-4 posts. Podriamos dejarlo con un número definido de scrolls
#   2. Si al hacer 3 scrolls seguidos no se cargan nuevos posts, hacer un break
#   3. Si tuvieramos el número de posts en algún lado en la página; pudiéramos hacer scroll hasta estar cercanos a ese número
#      esto lo podriamos hacer por ejemplo en Youtube
n_scrolls = 0
max_scrolls = 75
max_posts = 15
posts = driver.find_elements(by=By.XPATH, value='//div[@aria-describedby and @role="article"]')

while len(posts) < max_posts and n_scrolls < max_scrolls:
    hacer_scrolling_suavizado(driver, n_scrolls)
    posts = driver.find_elements(by=By.XPATH, value='//div[@aria-describedby and @role="article"]')
    n_scrolls += 1
    print('Termino scrolling: durmiendo')
    sleep(2)

# Volvemos a buscar los posts, dado que despues de scrollear el numero de estos aumentó
#posts = driver.find_elements(by=By.XPATH, value='//div[@aria-describedby and @role="article"]')
posts = WebDriverWait(driver, 20).until(EC.presence_of_all_elements_located((By.XPATH, '//div[@aria-describedby and @role="article"]')))
print(len(posts))

# Inicializando el almacenamiento de la data
texto_post = []
tiempo_post = []
reacciones_post = []  

for post in posts:
    # ME ARROJA ERROR 'texto_post', dice que no existe el Elemento. Consultar esto!!!
    try:
        texto_post.append(post.find_element(By.XPATH, './/div[class="xdj266r x11i5rnm xat24cr x1mh8g0r x1vvkbs x126k92a"]').text)
    except NoSuchElementException:
        texto_post.append('NaN')

    tiempo_post.append(post.find_element(by=By.XPATH, value='.//span[@id]//a[@aria-label]').text)
    # Este XPATH es unico dentro del post
    reacciones_post.append(post.find_element(By.XPATH, './/span[@class="x1e558r4"]').text)

driver.quit()

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df = pd.DataFrame({'tiempo_post': tiempo_post, 'reacciones': reacciones_post, 'texto_post':texto_post})
df.to_csv('comentarios.csv', index=False)


Termino scrolling: durmiendo
Termino scrolling: durmiendo
Termino scrolling: durmiendo
Termino scrolling: durmiendo
Termino scrolling: durmiendo
16
